# **1. 스팸 메일 분류기**

## **1-1. 데이터 다운로드**

> **데이터 다운로드**

In [ ]:
!curl -k -O https://www2.aueb.gr/users/ion/data/enron-spam/preprocessed/enron1.tar.gz

> **데이터 압축 해제**

In [ ]:
! tar -xf enron1.tar.gz enron1

> **압축 해제된 디렉토리 내용 확인**

In [ ]:
! ls -la enron1/

In [ ]:
! cat enron1/Summary.txt

In [ ]:
! ls -1 enron1/ham/*.txt | wc -l # # of non-spam email

In [ ]:
! ls -1 enron1/spam/*.txt | wc -l # # of spam email

In [ ]:
! ls enron1/ham

In [ ]:
! tail enron1/ham/0007.1999-12-14.farmer.ham.txt

## **1-2. 데이터 전처리**

> **메일 데이터 로드 및 레이블링**

*   `enron1/spam` 디렉토리의 모든 파일을 읽어 `emails` 리스트에 추가하고, 스팸임을 나타내는 `1`을 `labels` 리스트에 추가
*   `enron1/ham` 디렉토리의 모든 파일을 읽어 `emails` 리스트에 추가하고, 정상 메일임을 나타내는 `0`을 `labels` 리스트에 추가

In [ ]:
import glob, os


emails, labels = [], []
parition = 0

file_path = 'enron1/spam'

for fname in glob.glob(os.path.join(file_path, '*.txt')):
    with open(fname, 'r', encoding='ISO-8859-1') as f:
      emails.append(f.read())
      labels.append(1)

file_path = 'enron1/ham'
for fname in glob.glob(os.path.join(file_path, '*.txt')):
    with open(fname, 'r', encoding='ISO-8859-1') as f:
      emails.append(f.read())
      labels.append(0)

In [ ]:
print('number of emails = {}\nnumber of labels = {}'.format(len(emails), len(labels)))

In [ ]:
print(emails[1])

> **텍스트 데이터 정제**

In [ ]:
# 주어진 단어가 알파벳으로만 구성되어 있는지 확인
def letters_only(word):
  return word.isalpha()

In [ ]:
import nltk
nltk.download('names')   # 일반적인 이름(고유명사)
nltk.download('wordnet') # 단어의 기본형(어간)

In [ ]:
# 영어 이름 단어를 불러와 set 자료형으로 저장
from nltk.corpus import names
all_names = set(names.words())

In [ ]:
# 단어의 기본형(어간)을 찾아주는 Lemmatizer 객체 초기화 (예: running > run)
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [ ]:
def clean_text(doc):
  cleaned_doc = []
  for word in doc.split(' '): # split doc. by blank (' ')
    word = word.lower() # 소문자로 변환: ABD -> abd

    if letters_only(word) and word not in all_names and len(word) > 2: # remove number and punc. and name entity
      cleaned_doc.append(lemmatizer.lemmatize(word))
  return ' '.join(cleaned_doc)

cleaned_emails = [clean_text(doc) for doc in emails]

In [ ]:
cleaned_emails[1]

> **데이터셋 분할**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(cleaned_emails, labels, test_size=0.33, random_state=0)

> **텍스트 데이터를 수치형으로 변환**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(stop_words='english', max_features=500)
term_docs_train = cv.fit_transform(X_train)
term_docs_test = cv.transform(X_test)
print(term_docs_train[0])

In [ ]:
feature_names = cv.get_feature_names_out()
print(feature_names[195])
print(feature_names[284])

## **1-3. Naive Bayes 모델 훈련**

In [ ]:
from sklearn.naive_bayes import MultinomialNB
clf = MultinomialNB(alpha=1.0, fit_prior=True) # alpha: smoothing
clf.fit(term_docs_train, Y_train)

> **Test 예측**

In [ ]:
# [정상일 확률, 스팸일 확률]
prediction_prob = clf.predict_proba(term_docs_test)
prediction_prob[0:10]

In [ ]:
prediction = clf.predict(term_docs_test)
prediction[:10]

> **정확도 평가**

In [ ]:
accuracy = clf.score(term_docs_test, Y_test)
print('The accuracy using MultinomialNB is: {0:.1f}%'.format(accuracy*100))